In [0]:
from pyspark.sql.functions import sha2, concat_ws, col, lit, current_date

# 1. Load your cleaned Silver data
silver_df = spark.read.table("practice.silver.netflix_titles_cleaned")

# 2. Create the Hashing Logic
# We use '||' as a separator to prevent "hash collisions"
gold_prep_df = silver_df.withColumn("show_key", sha2(col("show_id"), 256)) \
    .withColumn("hash_diff", sha2(
        concat_ws("||", 
            col("director"), 
            col("cast"), 
            col("country"), 
            col("rating"), 
            col("release_year")
        ), 256)
    )

# 3. Add SCD Type 2 Metadata columns
gold_final_df = gold_prep_df.withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None).cast("date")) \
    .withColumn("is_current", lit(True))


# Preview the hashes
display(gold_final_df.select("show_id", "show_key", "hash_diff").limit(5))

In [0]:
gold_final_df.createOrReplaceTempView("incoming_data")


In [0]:
%sql
CREATE TABLE IF NOT EXISTS practice.gold.dim_shows (
    -- Surrogate Key & Change Detection
    show_key STRING,       -- Hash of show_id
    hash_diff STRING,      -- Hash of all descriptive columns

    -- Core Data
    show_id STRING,
    type STRING,
    title STRING,
    director STRING,
    cast STRING,
    country STRING,
    release_year INT,
    rating STRING,
    duration STRING,
    listed_in STRING,
    description STRING,
    date_added DATE,       -- Using your date_added_converted column

    -- SCD Type 2 Metadata
    start_date DATE,       -- When this version began
    end_date DATE,         -- When this version ended (NULL if current)
    is_current BOOLEAN,    -- TRUE for the latest version
    _ingested_at TIMESTAMP -- Audit trail from Bronze
)
USING DELTA;

In [0]:
%sql
-- This MERGE handles both New Inserts and Updates (SCD Type 2)
MERGE INTO practice.gold.dim_shows AS target
USING (
  -- We identify which records are actually 'new' or 'changed'
  SELECT 
    source.*,
    target.show_key AS merge_key -- This helps us identify if the record exists
  FROM incoming_data AS source
  LEFT JOIN practice.gold.dim_shows AS target
    ON source.show_id = target.show_id AND target.is_current = true
) AS staged_updates
ON target.show_key = staged_updates.merge_key

-- 1. If the show_id exists but the data (hash_diff) has changed:
-- We 'expire' the old record
WHEN MATCHED AND target.hash_diff <> staged_updates.hash_diff THEN
  UPDATE SET 
    target.is_current = false, 
    target.end_date = current_date()

-- 2. If the show_id is brand new:
-- We insert it as the current version
WHEN NOT MATCHED THEN
  INSERT (
    show_key, hash_diff, show_id, type, title, director, cast, 
    country, release_year, rating, duration, listed_in, 
    description, date_added, start_date, end_date, is_current
  )
  VALUES (
    staged_updates.show_key, staged_updates.hash_diff, staged_updates.show_id, 
    staged_updates.type, staged_updates.title, staged_updates.director, 
    staged_updates.cast, staged_updates.country, staged_updates.release_year, 
    staged_updates.rating, staged_updates.duration, staged_updates.listed_in, 
    staged_updates.description, staged_updates.date_added_converted, 
    current_date(), NULL, true
  );

In [0]:
%sql
INSERT INTO practice.gold.dim_shows
SELECT 
    source.show_key, source.hash_diff, source.show_id, source.type, source.title, 
    source.director, source.cast, source.country, source.release_year, 
    source.rating, source.duration, source.listed_in, source.description, 
    source.date_added_converted, current_date(), NULL, true, current_timestamp()
FROM incoming_data source
WHERE NOT EXISTS (
    SELECT 1 FROM practice.gold.dim_shows target 
    WHERE target.show_id = source.show_id AND target.is_current = true
);

In [0]:
%sql
SELECT * FROM practice.gold.dim_shows WHERE show_id = 's1'